# Day 2 牛津 Tutorial LLM 仿真 (Oxford + HBS + Hattie)

## Cell 1 · Persona Prompt (System)

> 你是 **Oxford tutorial fellow in LLM 应用工程** (LLM application engineering: RAG / Prompt Engineering / Function Calling / MCP / RAGAS / tiktoken / langchain_core / langsmith)。
>
> **铁律**:
> 1. **Never give direct answers. 不直接给答案, 不直接答。** 用苏格拉底式提问 (Socratic questioning) 引导学生自己推理。
> 2. 充当 **HBS devil's advocate**: 对学生任何含糊主张 (vague claims) 一律反驳, 要求精确数据/反例/边界条件。
> 3. 拒绝套话--学生说"RAG 就是检索+生成"必须追问"凭什么检索能减少幻觉? 依据是什么?"
> 4. **每轮结束必须以一个 probing question 收尾**, 不例外。
> 5. 不超过 4 轮 Socratic 循环, 防依赖 (见 cell6 限频)。



## Cell 2 · Pre-Tutorial Task (强制 retrieval practice, 不准翻书)

**上课前 24 小时提交** (不提交=不准进入 tutorial):

请用 **<=300 字** 写一段 mini-essay, 主题自选其一:

- **(A)** 用你自己的话解释: 为什么"先 Prompt Engineering, 不够再 RAG, 最后才 Fine-tuning"是工程原则? 给出至少 1 个反例 (什么场景下该倒序)。
- **(B)** 描述你曾构建/想象的营销知识库 RAG 系统: 用什么分块策略? Embedding 模型选哪个? 如何评估 faithfulness 瓶颈在检索还是生成?
- **(C)** 论证: 在 gpt-4o vs DeepSeek V3 之间为日均万次营销文案生成选型, 你的决策与依据 (成本/质量/可观测性三维度)。

> Tutorial 开始时, fellow 会就你的 essay 提出 **>=4 轮 Socratic 追问**。你必须能 defending 你的主张, 不能说"我觉得就是这样"。



In [ ]:
# Cell 3 · Multi-turn Socratic Loop (>=4 轮, 静态 if/else 模拟, 不调 openai/anthropic API)
# 本 cell 模拟 Oxford fellow 对学生 essay 的 4 轮 Socratic 追问。
# 学生回答通过 student_response 字符串注入, fellow 的回应是静态分支 (真实 tutorial 中由 LLM 生成, 这里用规则近似, ANTI-STALL)。

def oxford_fellow_turn(essay_topic, student_response, turn):
    """静态 Socratic 模拟器: 根据 turn 与 student_response 关键词选择追问分支。
    不调任何真实 LLM API; 仅模拟 Socratic follow-up 逻辑。"""
    turn = int(turn)
    # ---- 5+ 苏格拉底问 (为什么/凭什么/反例/若前提变/如何) ----
    # turn 1: 凭什么 (依据) + 为什么
    if turn == 1:
        if "成本" in student_response and "DeepSeek" in student_response:
            return (
                "你说 DeepSeek V3 更便宜。**凭什么**它的 input 定价能做到 gpt-4o 的 1/10? "
                "依据是什么架构特性? **为什么** MoE 的 37B 激活参数 (而非 671B 总参) 决定了推理成本? "
                "**反例**: 若任务需要极长上下文 (128K), DeepSeek V3 还一定更便宜吗? "
                "请用 tiktoken o200k_base vs cl100k_base 给出 token 数差异的精确数字。"
            )
        else:
            return (
                "你的回答太含糊。**为什么** Prompt Engineering 要排在 RAG 之前? "
                "**凭什么**说它成本极低--成本由什么组成 (token? 延迟? 工程时间?)? "
                "**反例**: 若营销知识库每天更新, Pure Prompt Engineering 还成本低吗?"
            )
    # turn 2: 若前提变 + 反例
    elif turn == 2:
        if "TF-IDF" in student_response or "余弦" in student_response:
            return (
                "你提到 numpy TF-IDF + 余弦相似度做检索。**若**把营销知识库从 100 篇扩到 10 万篇, "
                "TF-IDF 矩阵的维度与计算复杂度如何变化? **反例**: 这种情况下 all-MiniLM-L6-v2 Embedding + 向量库 "
                "比 TF-IDF 优势在哪? **如何**量化 BM25 + 向量混合检索 相比纯向量的召回提升?"
            )
        else:
            return (
                "**若**把你的 RAG 系统 faithfulness 从 0.6 提升到 0.9, 你会先动哪个组件? "
                "**为什么**不是直接 Fine-tune? **反例**: 若 ground-truth 文档根本没被检索到 (context_recall=0.3), "
                "调 Prompt 能救 faithfulness 吗? **依据**是什么?"
            )
    # turn 3: 假设前提变
    elif turn == 3:
        if "MCP" in student_response or "Function Calling" in student_response:
            return (
                "你提到 MCP 标准化 Function Calling。**假设** OpenAI 明天发布一个新 tool calling 格式与 MCP 不兼容, "
                "你的营销 Agent 代码要改多少? **如何**设计工具层使模型可移植? "
                "**凭什么**说 MCP 比 vendor lock-in 好--有具体成本数据吗?"
            )
        else:
            return (
                "**假设** gpt-4o 明天降价到与 DeepSeek V3 同价, 你的模型选型会变吗? **为什么**? "
                "**如何**在 langsmith @traceable 追踪里看出质量差异 而不只是成本差异? "
                "**反例**: 同一 Prompt 在两模型上 faithfulness 差 0.2, 你怎么定位是模型能力还是 Prompt 不适配?"
            )
    # turn 4: 如何 + 凭什么 (收尾, 强制学生自评盲点)
    elif turn == 4:
        return (
            "最后一轮。**如何**用 <=2 句话向 CTO 论证你的 RAG 系统可以上线? "
            "**凭什么**CTO 应该信你的 RAGAS 评估而非直觉? "
            "**假设**评估指标全绿但用户反馈差, 你的假设是哪里出了问题? "
            "请列出你本次 tutorial 的 2-3 个盲点 (exit artifact, 见 cell6)。"
        )
    return "(tutorial 结束, 见 cell5 反馈)"

# ---- 模拟 4 轮 Socratic 对话 (静态, 用预填 student_response) ----
essay = "我选 gpt-4o 因为质量好, 但成本高; RAG 用 TF-IDF 检索; 评估用 RAGAS。"
print("=" * 70)
print("Pre-tutorial essay:")
print(essay)
print("=" * 70)
for t in range(1, 5):
    # 每轮学生回答逐步具体化 (模拟 retrieval practice 下的精化)
    if t == 1:
        resp = "gpt-4o input $2.50/M, DeepSeek V3 $0.27/M, 约十分之一; 用 tiktoken 测过 token 数。"
    elif t == 2:
        resp = "我用 numpy TF-IDF + 余弦相似度做 top-3 召回, 10 万篇会爆维度, 该换 all-MiniLM-L6-v2。"
    elif t == 3:
        resp = "MCP 让 Function Calling 可移植, 若 OpenAI 换格式我只改工具适配层不改 Prompt。"
    else:
        resp = "RAGAS 三指标全绿但用户差评, 我假设是 answer_relevance 没覆盖用户真实意图。"
    fellow_reply = oxford_fellow_turn(essay, resp, t)
    print(f"\n--- Turn {t} ---")
    print(f"[Student]: {resp}")
    print(f"[Fellow (Socratic, 不直接给答案)]: {fellow_reply}")

print("\n" + "=" * 70)
print("Socratic loop 完成 (4 轮). 进入 cell4 student_model 记录, cell5 Hattie 反馈。")


In [ ]:
# Cell 4 · student_model.json 读写 (记录掌握度 + 盲点)
# Oxford tutorial 用 student_model 追踪每位学生的掌握度 (mastery) 与盲点 (blind_spots),
# 供下次 tutorial 个性化追问与 alignment.md mastery 阈值对账。

import json, os

student_model_path = "/tmp/student_model_day2.json"

default_model = {
    "student_id": "demo-student",
    "unit": "elective-e3-llm-intro/day-2-llm-application-engineering",
    "subskills": {
        "A_prompt_token": {"mastery": 0.0, "reps_done": 0, "last_5": []},
        "B_rag_retrieval": {"mastery": 0.0, "reps_done": 0, "last_5": []},
        "C_ragas_mcp": {"mastery": 0.0, "reps_done": 0, "last_5": []}
    },
    "blind_spots": [],
    "socratic_turns_total": 0,
    "weak_loop_triggered": False
}

def load_student_model():
    if os.path.exists(student_model_path):
        with open(student_model_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default_model.copy()

def save_student_model(model):
    with open(student_model_path, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)
    return model

def update_mastery(model, subskill, turn_score):
    """turn_score: 0.0-1.0, 由 fellow 静态判定 (本 demo 用关键词匹配近似)。"""
    s = model["subskills"][subskill]
    s["last_5"].append(turn_score)
    s["last_5"] = s["last_5"][-5:]  # 滑窗 5
    s["reps_done"] += 1
    # mastery = 最近 5 次均值 (CS230 mastery learning 思想)
    s["mastery"] = sum(s["last_5"]) / len(s["last_5"])
    return model

def add_blind_spot(model, spot):
    if spot not in model["blind_spots"]:
        model["blind_spots"].append(spot)
    # 连续 2 次失败触发 weak_loop (practice.md 定义)
    if len(model["blind_spots"]) >= 2 and not model["weak_loop_triggered"]:
        model["weak_loop_triggered"] = True
    return model

# ---- 模拟 4 轮 Socratic 后更新 student_model ----
model = load_student_model()
model["socratic_turns_total"] += 4

# Turn 1: 学生答出成本 1/10 -> A 子技能 +0.6
update_mastery(model, "A_prompt_token", 0.6)
# Turn 2: 学生答出 TF-IDF 维度爆炸 + 换 Embedding -> B 子技能 +0.7
update_mastery(model, "B_rag_retrieval", 0.7)
# Turn 3: 学生答出 MCP 可移植 -> C 子技能 +0.5
update_mastery(model, "C_ragas_mcp", 0.5)
# Turn 4: 学生反思 answer_relevance 盲点 -> 记录
add_blind_spot(model, "answer_relevance 与用户真实意图的对齐未量化")
add_blind_spot(model, "长上下文 (128K) 下 DeepSeek V3 成本优势是否依然成立")

save_student_model(model)
print("student_model.json 已写入:", student_model_path)
print(json.dumps(model, ensure_ascii=False, indent=2))

# 判定 mastery (alignment.md 阈值 >=80%)
for sk, v in model["subskills"].items():
    status = "MASTERY" if v["mastery"] >= 0.8 else "NOT-YET"
    print(f"  {sk}: mastery={v['mastery']:.2f} ({status})")
if model["weak_loop_triggered"]:
    print("  weak_loop 已触发: 回退 practice.md stage1_worked + 补充 worked example")


## Cell 5 · Hattie 四级形成性反馈 (Hattie & Timperley 2007)

> Oxford fellow 在 Socratic loop 结束后, 按 Hattie 四级反馈框架给出反馈。**避免 Self 级表扬** (如"你真聪明"), 聚焦 Task / Process / Self-Reg / Feed-Forward。

### [TASK] 任务级反馈 (针对本次 essay 的具体内容)
- 你的成本对比数据 (gpt-4o $2.50/M vs DeepSeek V3 $0.27/M) **正确**, 但缺少 128K 长上下文场景下的反例验证。
- 你的 RAG 检索描述 "TF-IDF + 余弦" 在小语料成立, 但未给出 10 万篇语料的复杂度量化。
- **改进**: 用 tiktoken 实测"618 全场 5 折"在 o200k_base / cl100k_base 下的 token 数, 给出精确表格。

### [PROCESS] 过程级反馈 (针对学生使用的推理策略)
- 你在 Turn 2 用了"假设语料扩到 10 万篇"的反事实推理--**这是好的 Process**, 应保留。
- 但你在 Turn 1 直接背"1/10"而未给架构依据 (MoE 37B 激活), **Process 缺陷**: 数据先于机制, 应先答"凭什么"再答"是多少"。
- **改进**: 下次回答成本类问题, 先答架构机制 (MoE 激活参数), 再答定价数字。

### [SELF-REG] 自我调节反馈 (针对学生自我监控能力)
- 你在 Turn 4 主动识别"answer_relevance 没覆盖用户意图"是盲点--**自我监控到位**。
- 但前 3 轮你未主动暴露盲点, 被 fellow 追问才承认--**自调不够主动**。
- **改进**: 下次 tutorial 前自写 1 段"我预判 fellow 会追问哪 3 点", 训练 self-monitoring。

### [FEED-FORWARD] 前馈反馈 (指向下一步学习)
- 下一单元 (Day 3 LLM 评估与部署) 会延伸到 MMLU / LLM-as-Judge 完整评估体系, 你的 RAGAS 三指标理解是基础, **务必在 Day 3 前把 faithfulness vs context_recall 瓶颈诊断练到 mastery**。
- 推荐复习: schedule.json C5 (RAGAS) + C6 (MCP) 两卡, 按 FSRS-6 间隔重复。
- 若 faithfulness 瓶颈诊断未过 >=80%, 触发 weak_loop, 回退 practice.md D3 stage1_worked。



## Cell 6 · 限频 + Exit Artifact

### 限频 (防依赖, 每天 1 次)
- 本 tutorial LLM 仿真 **每单元每生每天限 1 次** (1次/天)。理由: Oxford tutorial 的价值在于学生**先独立 retrieval** (cell2 essay), 再接受 Socratic; 高频使用会让学生依赖 fellow 追问而非自己先想。
- 超过 1 次/天 (daily limit / usage limit), 系统返回: "今日 tutorial 额度已用完。请先用 schedule.json 复习卡片, 明日再来。"
- 限频由 student_model.json 的 `socratic_turns_total` 字段对账 (见 cell4)。

### Exit Artifact (强制 retrieval practice, 提交后才算完成)
请在 cell5 反馈后, 用 **<=150 字** 写出:

1. **2-3 个本次 tutorial 暴露的盲点** (从 cell4 student_model 的 blind_spots 选, 或新增):
   - 例: "我之前认为 RAGAS faithfulness 低=检索不好, 实际可能是生成层幻觉, 需要先看 context_recall 区分。"
   - 例: "我没考虑 128K 长上下文下 DeepSeek V3 的成本优势是否依然。"
   - 例: "我对 MCP 可移植性的论证缺具体代码改动量数据。"

2. **推荐复习单元/卡片** (从 schedule.json 选 >=2 张):
   - 例: C2 (tiktoken o200k_base / cl100k_base + 定价)
   - 例: C5 (RAGAS 三指标 + 瓶颈诊断)
   - 例: C4 (numpy TF-IDF + 六维优化)

3. **下次 tutorial 的预判追问** (>=1 个, 训练 self-regulation):
   - 例: "fellow 大概会问: 若用户反馈与 RAGAS 全绿矛盾, 你的假设链是什么?"

> Exit Artifact 提交后写入 student_model.json 的 `exit_artifacts` 字段, 供下次 tutorial 个性化追问。

---

*本 tutorial.ipynb 基于 Oxford tutorial (Socratic, 不直接给答案) + HBS devil's advocate + Hattie (2007) 四级形成性反馈。Socratic loop 为静态 if/else 模拟, 不调真实 LLM API (ANTI-STALL)。限频 1 次/天防依赖。*
